<a href="https://colab.research.google.com/github/srinivasa04/UKVisaAIAssistant/blob/main/UK_Visa_Guidance_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# # UK Visa Guidance Agent

This notebook implements an AI agent that assists immigration consultants
by retrieving information from official GOV.UK visa guidance.

Features

• Gemini LLM
• RAG
• ChromaDB
• Planning
• Tool Calling
• Memory
• Adaptive Behaviour

In [48]:
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf \
    google-genai

In [49]:
# ==========================================
# Standard Python Libraries
# ==========================================
import os
import time
import json
import logging

# ==========================================
# Google Gemini
# ==========================================
from google import genai
from google.colab import userdata

# ==========================================
# LangChain - Document Loading
# ==========================================
from langchain_community.document_loaders import PyPDFLoader

# ==========================================
# LangChain - Text Splitting
# ==========================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma

# ==========================================
# LangChain - Chroma Vector Store
# ==========================================
from langchain_chroma import Chroma


In [50]:
# Configure the API Key

from google.colab import userdata
from google import genai

client = genai.Client(
    api_key=userdata.get("GOOGLE_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Hello"
)

print(response.text)

Hello! How can I help you today?


In [51]:
#Create data folder to store documents
import os

# Create the 'data' folder if it doesn't already exist
os.makedirs("data", exist_ok=True)

print("Created 'data' folder.")

Created 'data' folder.


In [52]:
# Load the pdf documents from data folder
from langchain_community.document_loaders import PyPDFLoader
import os

documents = []

for file in os.listdir("data"):

    if file.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join("data", file))

        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages.")

Loaded 371 pages.


In [53]:
# Split into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 1097


In [54]:
texts = [doc.page_content for doc in chunks]
metadata = [doc.metadata for doc in chunks]

print(f"Total chunks: {len(texts)}")

Total chunks: 1097


In [55]:
import time
from google.genai.errors import ClientError

EMBEDDING_MODEL = "gemini-embedding-001"

def embed_batch(batch):

    while True:
        try:
            response = client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=batch
            )

            return [embedding.values for embedding in response.embeddings]

        except ClientError as e:

            if "RESOURCE_EXHAUSTED" in str(e):

                print("Rate limit reached. Waiting 30 seconds...")
                time.sleep(30)

            else:
                raise

In [56]:
def embed_documents(texts, batch_size=100):

    embeddings = []
    total = len(texts)

    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        batch_vectors = embed_batch(batch)
        embeddings.extend(batch_vectors)
        print(f"Completed {min(i+batch_size,total)}/{total}")
        time.sleep(2)

    return embeddings

In [57]:
embeddings = embed_documents(texts)

Completed 100/1097
Completed 200/1097
Completed 300/1097
Completed 400/1097
Completed 500/1097
Completed 600/1097
Completed 700/1097
Completed 800/1097
Completed 900/1097
Completed 1000/1097
Completed 1097/1097


In [58]:
print(len(embeddings))
print(len(embeddings[0]))

1097
3072


In [59]:
import chromadb

client_db = chromadb.PersistentClient(path="./chroma_db")

collection = client_db.get_or_create_collection(
    name="uk_visa_guidance"
)

In [60]:
ids = [str(i) for i in range(len(texts))]

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadata
)

In [61]:
question = "What documents are required for a Skilled Worker visa?"

query_vector = embed_batch([question])[0]

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):

    print("="*80)
    print(f"Result {i+1}")
    print("="*80)

    print(doc[:700])

Result 1
Validity requirements for a Skilled
Worker
SW 1.1. A person applying for entry clearance or
permission to stay as a Skilled Worker must apply
online on the gov.uk website on the specified form
as follows:
(a) for applicants outside the UK, form “Skilled
Worker visa”; or
(b) for applicants inside the UK, form “Skilled
Worker”.
SW 1.2. An application for entry clearance or
permission to stay as a Skilled Worker must meet
all the following requirements:
(a) any fee and Immigration Health Charge must
have been paid; and
(b) the applicant must have provided biometrics
when required; and
(c) the applicant must have provided a passport
or other travel document which satisfactorily
establishes their
Result 2
7. Documents you'll need to apply
When you apply you’ll need to provide:
your certificate of sponsorship reference number - your employer will give
you this
proof of your knowledge of English (/skilled-worker-visa/knowledge-of-english)
a valid passport or other document that shows

In [62]:
# Send retrieved context to Gemini

context = "\n\n".join(results["documents"][0])

prompt = f"""
You are a UK Visa Guidance Assistant.

Answer ONLY using the context below.

If the answer cannot be found,
say you could not find it.

Context:

{context}

Question:

{question}
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

Based on the provided context, when you apply for a Skilled Worker visa, you will need to provide:

**Required documents/information:**
* A valid passport or other travel document that shows/establishes your identity and nationality
* Your certificate of sponsorship reference number (provided by your employer)
* Proof of your knowledge of English
* Your job title and annual salary
* Your job’s occupation code
* The name of your employer and their sponsor licence number (found on your certificate of sponsorship)

**Other documents you might need (depending on your circumstances):**
* Evidence that you have enough personal savings to support yourself


# Tool calling

                      User Question
                          │
                          ▼
                Intent / Tool Selector
                  │                │
         Eligibility Tool     Document Search Tool
                  │                │
                  └──────┬─────────┘
                         ▼
                    Gemini 3.6 Flash
                         ▼
                  Final Response

In [63]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GOOGLE_API_KEY")
)

MODEL_NAME = "gemini-3.6-flash"

In [64]:
# Search document tool

def search_documents(question):

    query_vector = embed_batch([question])[0]

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=3
    )

    if len(results["documents"]) == 0:
        return "No relevant GOV.UK guidance found."

    docs = results["documents"][0]

    if len(docs) == 0:
        return "No relevant GOV.UK guidance found."

    return "\n\n".join(docs)

In [65]:
# Basic eligibility tool

def check_basic_eligibility(user_text):

    text = user_text.lower()

    checks = {
        "sponsor": "sponsor" in text,
        "english": "english" in text,
        "passport": "passport" in text
    }

    missing = [
        key
        for key, value in checks.items()
        if not value
    ]

    if len(missing) == 0:

        return (
            "Based on the information provided, "
            "the applicant appears to satisfy the basic "
            "requirements checked by this tool. "
            "This is not legal advice and does not guarantee visa approval."
        )

    return (
        "Potential missing requirements: "
        + ", ".join(missing)
    )

# Planning, Memory & Context

In [66]:
import json

def create_plan(question):

    prompt = f"""
You are the planner for a UK Visa Guidance Agent.

Available actions

1. search_documents
   Use for:
   - visa documents
   - requirements
   - immigration rules
   - financial evidence
   - processing guidance

2. check_basic_eligibility
   Use when the user provides personal circumstances.

3. answer_directly
   Use only when no tools are required.

You may choose MULTIPLE actions.

Examples

Question:
"I have a sponsor and passport. What documents do I need?"

Output:

{{
    "steps":[
        "check_basic_eligibility",
        "search_documents",
        "answer_directly"
    ]
}}

Question:
"{question}"

Return ONLY valid JSON.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    if text.startswith("```"):
        text = text.split("\n",1)[1]
        text = text.rsplit("```",1)[0]

    return json.loads(text)

In [67]:
ALLOWED_STEPS = {
    "search_documents",
    "check_basic_eligibility",
    "answer_directly"
}

plan = create_plan(question)
plan["steps"] = [s for s in plan["steps"] if s in ALLOWED_STEPS]

In [68]:
# Add Short-Term Memory

conversation_memory = []

def remember(question, answer, tool):

    conversation_memory.append({
        "question": question,
        "answer": answer,
        "tool": tool
    })

    if len(conversation_memory) > 5:
        conversation_memory.pop(0)

In [69]:
def build_memory():

    if len(conversation_memory) == 0:

        return "No previous conversation."

    history = ""

    for item in conversation_memory:

        history += f"""
        User:
        {item['question']}

        Assistant:
        {item['answer']}

        """

    return history

In [70]:
def clear_memory():

    conversation_memory.clear()

    print("Conversation memory cleared.")

# Adaptive Behaviour

In [71]:
feedback_memory = []

def store_feedback(question,
                   answer,
                   rating,
                   comments=""):

    feedback_memory.append({

        "question": question,

        "answer": answer,

        "rating": rating,

        "comments": comments

    })

    print("Feedback stored.")

In [72]:
store_feedback(

    question="What documents are needed?",

    answer="Student visa requires ...",

    rating="negative",

    comments="Please answer more briefly."
)

Feedback stored.


In [73]:
user_preferences  = {

    "response_length":"normal",

    "cite_sources":True,

    "use_bullets":False
}

In [74]:
def update_behaviour():

    for feedback in feedback_memory:

        comments = feedback["comments"].lower()

        if "short" in comments or "brief" in comments:

            user_preferences["response_length"] = "short"

        if "bullet" in comments:

            user_preferences["use_bullets"] = True

        if "source" in comments:

            user_preferences["cite_sources"] = True

In [76]:
import logging

logger = logging.getLogger("VisaAgent")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

# Console output
console = logging.StreamHandler()
console.setFormatter(formatter)

# File output
file = logging.FileHandler("visa_agent.log")
file.setFormatter(formatter)

logger.addHandler(console)
logger.addHandler(file)

In [92]:
# Define the agent with planning
import time

tool_result = ""

def visa_agent(question):

    if not question.strip():
      return ("Please enter a question about UK visas.")

    logger.info(f"Question: {question}")

    start_time = time.time()
    try:
        plan = create_plan(question)
        print("=" * 70)
        print("PLAN")
        print(plan)
        print("=" * 70)

        memory = build_memory()
        tool_outputs = []
        tools_used = []

        for step in plan["steps"]:

            if step == "check_basic_eligibility":
                result = check_basic_eligibility(question)
                tool_outputs.append(
                    "Eligibility Check:\n" + result
                )
                tools_used.append(step)

            elif step == "search_documents":
                result = search_documents(question)
                tool_outputs.append(
                    "Document Search:\n" + result
                )
                tools_used.append(step)

            elif step == "answer_directly":
                pass

        combined_output = "\n\n".join(tool_outputs)

        update_behaviour()

        style = f"""

        Response Length:
        {user_preferences['response_length']}

        Use Bullets:
        {user_preferences['use_bullets']}

        Cite Sources:
        {user_preferences['cite_sources']}

        """

        prompt = f"""
    You are a UK Visa Guidance Assistant.

    Behaviour Settings
    {style}

    Previous Conversation
    {memory}

    Tool Results
    {combined_output}

    Current Question
    {question}

    Instructions

    1. Use ALL tool outputs.

    2. If eligibility was checked,
    mention that it is only an initial assessment.

    3. If document search returned
    "No relevant GOV.UK guidance found",
    say so.

    4. Never invent visa rules.

    5. Produce one final answer.
    """

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt)

        answer = response.text

        conversation_memory.append({
            "question": question,
            "answer": answer,
            "tools": tools_used
        })

        if len(conversation_memory) > 5:
            conversation_memory.pop(0)

        logger.info(f"Answer: {answer}")
        latency = round(time.time() - start_time, 2)
        print(f"Response Time: {latency} seconds")
        return answer

    except Exception as e:
        logger.error(str(e))
        return (
        "The language model is currently unavailable. "
        "Please try again shortly."
        )

# Demonstration

In [78]:
visa_agent("What documents are required for a Skilled Worker visa?")

2026-07-25 20:32:07,131 | INFO | Question: What documents are required for a Skilled Worker visa?
INFO:VisaAgent:Question: What documents are required for a Skilled Worker visa?


PLAN
{'steps': ['search_documents', 'answer_directly']}


2026-07-25 20:32:17,216 | INFO | Answer: Based on official GOV.UK guidance, here are the documents and information required when applying for a Skilled Worker visa:

### Standard Required Documents and Details
* **Valid Passport or Travel Document:** Must satisfactorily establish your identity and nationality.
* **Certificate of Sponsorship (CoS) Reference Number:** Provided by your UK employer.
* **Employer Information:** Your employer's name and sponsor licence number (found on your CoS).
* **Job Details:** Your job title, annual salary, and occupation code.
* **Proof of Knowledge of English:** Evidence meeting the English language requirement.

---

### Additional Documents You Might Need (Depending on Circumstances)
* **Proof of Personal Savings:** Evidence showing you have sufficient personal savings to support yourself in the UK.
* **Copy of Certificate of Sponsorship:** You can ask your employer for a copy if needed.

---

### Additional Application Requirements
As part of the a

Response Time: 10.09 seconds


"Based on official GOV.UK guidance, here are the documents and information required when applying for a Skilled Worker visa:\n\n### Standard Required Documents and Details\n* **Valid Passport or Travel Document:** Must satisfactorily establish your identity and nationality.\n* **Certificate of Sponsorship (CoS) Reference Number:** Provided by your UK employer.\n* **Employer Information:** Your employer's name and sponsor licence number (found on your CoS).\n* **Job Details:** Your job title, annual salary, and occupation code.\n* **Proof of Knowledge of English:** Evidence meeting the English language requirement.\n\n---\n\n### Additional Documents You Might Need (Depending on Circumstances)\n* **Proof of Personal Savings:** Evidence showing you have sufficient personal savings to support yourself in the UK.\n* **Copy of Certificate of Sponsorship:** You can ask your employer for a copy if needed.\n\n---\n\n### Additional Application Requirements\nAs part of the application process, yo

In [79]:
visa_agent("I have a sponsor and passport. Am I eligible?")

2026-07-25 20:32:17,227 | INFO | Question: I have a sponsor and passport. Am I eligible?
INFO:VisaAgent:Question: I have a sponsor and passport. Am I eligible?


PLAN
{'steps': ['check_basic_eligibility', 'search_documents', 'answer_directly']}


2026-07-25 20:32:28,604 | INFO | Answer: Based on the information provided, here is an **initial assessment** of your eligibility for a Skilled Worker visa. Please note that this is only an initial assessment and not a final decision.

While having a valid passport and a UK sponsor (Certificate of Sponsorship) are essential first steps, having these alone does not automatically make you eligible. 

### Potential Missing Requirements
Based on official GOV.UK rules and an initial check:

1. **Proof of Knowledge of English:** 
   * The eligibility check highlights **English language proficiency** as a potential missing requirement. You must satisfy the English language requirement (e.g., through a certified test, a degree taught in English, or being from an exempt country).

2. **Certificate of Sponsorship (CoS) Validity:**
   * Your CoS must have been issued by your sponsor **no more than 3 months (or 90 days)** before the date you submit your visa application.

3. **Age Requirement:**
 

Response Time: 11.38 seconds


'Based on the information provided, here is an **initial assessment** of your eligibility for a Skilled Worker visa. Please note that this is only an initial assessment and not a final decision.\n\nWhile having a valid passport and a UK sponsor (Certificate of Sponsorship) are essential first steps, having these alone does not automatically make you eligible. \n\n### Potential Missing Requirements\nBased on official GOV.UK rules and an initial check:\n\n1. **Proof of Knowledge of English:** \n   * The eligibility check highlights **English language proficiency** as a potential missing requirement. You must satisfy the English language requirement (e.g., through a certified test, a degree taught in English, or being from an exempt country).\n\n2. **Certificate of Sponsorship (CoS) Validity:**\n   * Your CoS must have been issued by your sponsor **no more than 3 months (or 90 days)** before the date you submit your visa application.\n\n3. **Age Requirement:**\n   * You must be **18 or ov

In [80]:
visa_agent("What about my wife?")

2026-07-25 20:32:28,615 | INFO | Question: What about my wife?
INFO:VisaAgent:Question: What about my wife?


PLAN
{'steps': ['check_basic_eligibility', 'search_documents', 'answer_directly']}


2026-07-25 20:32:43,043 | INFO | Answer: Please note that an automated eligibility check was performed, but this is **only an initial assessment** (and highlights standard core requirements to verify, such as your sponsor, English language proficiency, and passport details).

Based on official GOV.UK guidance, here is how the rules apply to your wife (as your dependant):

### General Rules for Partners
* **Application Timing:** Your wife can apply at the same time as your main application, or at any time before her current visa expires (if she is already in the UK).
* **Fees:** She will need to pay the relevant visa fees and charges for her dependant visa type.

---

### Specific Rules Based on Job Type and Location

#### 1. If applying from **outside the UK**:
She can apply to join you as your dependant. However, if you are sponsored as a **care worker or senior care worker**, she can only join you if:
* You have been continually employed as a care worker or senior care worker in the 

Response Time: 14.43 seconds


'Please note that an automated eligibility check was performed, but this is **only an initial assessment** (and highlights standard core requirements to verify, such as your sponsor, English language proficiency, and passport details).\n\nBased on official GOV.UK guidance, here is how the rules apply to your wife (as your dependant):\n\n### General Rules for Partners\n* **Application Timing:** Your wife can apply at the same time as your main application, or at any time before her current visa expires (if she is already in the UK).\n* **Fees:** She will need to pay the relevant visa fees and charges for her dependant visa type.\n\n---\n\n### Specific Rules Based on Job Type and Location\n\n#### 1. If applying from **outside the UK**:\nShe can apply to join you as your dependant. However, if you are sponsored as a **care worker or senior care worker**, she can only join you if:\n* You have been continually employed as a care worker or senior care worker in the UK on a Skilled Worker vis

In [81]:
visa_agent("Please answer briefly.")

2026-07-25 20:32:43,055 | INFO | Question: Please answer briefly.
INFO:VisaAgent:Question: Please answer briefly.


PLAN
{'steps': ['answer_directly']}


2026-07-25 20:32:50,489 | INFO | Answer: Please note that any eligibility check mentioned is **only an initial assessment**.

### Brief Summary for Your Wife (Dependant)

* **Application:** She can apply at the same time as you, or before her current UK visa expires if she is already in the UK.
* **Fees:** She must pay the relevant visa application fee and Immigration Health Charge (IHC).
* **Care Worker Restriction:** If your job is a care worker or senior care worker, she can generally only join or switch as your dependant if you have been continuously employed in the UK in that role on a Skilled Worker visa since before **11 March 2024**.
INFO:VisaAgent:Answer: Please note that any eligibility check mentioned is **only an initial assessment**.

### Brief Summary for Your Wife (Dependant)

* **Application:** She can apply at the same time as you, or before her current UK visa expires if she is already in the UK.
* **Fees:** She must pay the relevant visa application fee and Immigrati

Response Time: 7.43 seconds


'Please note that any eligibility check mentioned is **only an initial assessment**.\n\n### Brief Summary for Your Wife (Dependant)\n\n* **Application:** She can apply at the same time as you, or before her current UK visa expires if she is already in the UK.\n* **Fees:** She must pay the relevant visa application fee and Immigration Health Charge (IHC).\n* **Care Worker Restriction:** If your job is a care worker or senior care worker, she can generally only join or switch as your dependant if you have been continuously employed in the UK in that role on a Skilled Worker visa since before **11 March 2024**.'

In [82]:
visa_agent("What documents do I need?")

2026-07-25 20:32:50,500 | INFO | Question: What documents do I need?
INFO:VisaAgent:Question: What documents do I need?


PLAN
{'steps': ['search_documents', 'answer_directly']}


2026-07-25 20:33:03,141 | INFO | Answer: Please note that any eligibility check is **only an initial assessment**.

Based on the latest official GOV.UK guidance retrieved, here are the documents you may need to provide depending on your circumstances:

### 1. Travel and Identification Documents
* **Valid Passport/Travel Document:** Your current passport.
* **Previous Passports:** Copies of previous passports showing evidence of travel to other countries.
* **Proof of Legal Residence:** Confirmation of legal residence if you are applying from a country where you are not a national (if your right to reside is not included in your passport).

### 2. Financial and Employment Documents
* **Financial Evidence:** Bank statements or building society books detailing the origin of the funds held to show you have sufficient funds available.
* **Proof of Earnings / Employment:** A letter from your employer confirming your employment details (start date, salary, role, and company contact details).


Response Time: 12.64 seconds


'Please note that any eligibility check is **only an initial assessment**.\n\nBased on the latest official GOV.UK guidance retrieved, here are the documents you may need to provide depending on your circumstances:\n\n### 1. Travel and Identification Documents\n* **Valid Passport/Travel Document:** Your current passport.\n* **Previous Passports:** Copies of previous passports showing evidence of travel to other countries.\n* **Proof of Legal Residence:** Confirmation of legal residence if you are applying from a country where you are not a national (if your right to reside is not included in your passport).\n\n### 2. Financial and Employment Documents\n* **Financial Evidence:** Bank statements or building society books detailing the origin of the funds held to show you have sufficient funds available.\n* **Proof of Earnings / Employment:** A letter from your employer confirming your employment details (start date, salary, role, and company contact details).\n* **Self-Employment Evidence

In [83]:
visa_agent("What's the weather today?")

2026-07-25 20:33:03,152 | INFO | Question: What's the weather today?
INFO:VisaAgent:Question: What's the weather today?


PLAN
{'steps': ['answer_directly']}


2026-07-25 20:33:09,202 | INFO | Answer: I am a UK Visa Guidance Assistant, so I do not have access to real-time weather information or daily weather forecasts. Please check a local weather website or app for today's forecast. 

If you have any further questions regarding UK visas, application requirements, or eligibility, please let me know!
INFO:VisaAgent:Answer: I am a UK Visa Guidance Assistant, so I do not have access to real-time weather information or daily weather forecasts. Please check a local weather website or app for today's forecast. 

If you have any further questions regarding UK visas, application requirements, or eligibility, please let me know!
2026-07-25 20:33:09,204 | INFO | Latency: 6.05 seconds
INFO:VisaAgent:Latency: 6.05 seconds


Response Time: 6.05 seconds


"I am a UK Visa Guidance Assistant, so I do not have access to real-time weather information or daily weather forecasts. Please check a local weather website or app for today's forecast. \n\nIf you have any further questions regarding UK visas, application requirements, or eligibility, please let me know!"

In [87]:
print(visa_agent("I have a sponsor, passport and English qualification. Am I eligible and what documents do I need? "))

2026-07-25 21:17:39,443 | INFO | Question: I have a sponsor, passport and English qualification. Am I eligible and what documents do I need? 
INFO:VisaAgent:Question: I have a sponsor, passport and English qualification. Am I eligible and what documents do I need? 


PLAN
{'steps': ['check_basic_eligibility', 'search_documents', 'answer_directly']}


2026-07-25 21:17:49,637 | INFO | Answer: Please note that any eligibility check mentioned is **only an initial assessment**. Based on the information provided, you appear to satisfy the basic requirements checked by our tool. However, this is not legal advice and does not guarantee visa approval.

---

### Eligibility Assessment
Having a valid passport, a sponsor (Certificate of Sponsorship), and proof of your English qualification covers several core requirements for a Skilled Worker visa. To be fully eligible, you must also satisfy the following rules based on official GOV.UK guidance:

* **Age Requirement (SW 1.3):** You must be aged **18 or over** on the date of application.
* **Certificate of Sponsorship (CoS) Timing:** Your CoS must have been issued by your sponsor no more than **3 months (or 90 days)** before the date of application.
* **Government/International Scholarship Consent (SW 1.4):** If you received an award covering both fees and living costs from a government or inte

Response Time: 10.19 seconds
Please note that any eligibility check mentioned is **only an initial assessment**. Based on the information provided, you appear to satisfy the basic requirements checked by our tool. However, this is not legal advice and does not guarantee visa approval.

---

### Eligibility Assessment
Having a valid passport, a sponsor (Certificate of Sponsorship), and proof of your English qualification covers several core requirements for a Skilled Worker visa. To be fully eligible, you must also satisfy the following rules based on official GOV.UK guidance:

* **Age Requirement (SW 1.3):** You must be aged **18 or over** on the date of application.
* **Certificate of Sponsorship (CoS) Timing:** Your CoS must have been issued by your sponsor no more than **3 months (or 90 days)** before the date of application.
* **Government/International Scholarship Consent (SW 1.4):** If you received an award covering both fees and living costs from a government or international sc

# Evaluation & Engineering Review Coding

In [84]:
evaluation_questions = [

    # -----------------------------
    # Skilled Worker
    # -----------------------------

    "What documents are required for a Skilled Worker visa?",

    "Can my spouse come with me on a Skilled Worker visa?"
    # -----------------------------
    # Student
    # -----------------------------

      "Can I work while studying in the UK?",

    # -----------------------------
    # Visitor
    # -----------------------------

    "How long can I stay on a Visitor visa?",

    # -----------------------------
    # Eligibility
    # -----------------------------

    "I have a sponsor and passport. Am I eligible?",

    # -----------------------------
    # Memory
    # -----------------------------

    "What documents are needed?",


    # -----------------------------
    # Out-of-domain
    # -----------------------------

    "What's the weather in London?",

    # -----------------------------
    # Unknown
    # -----------------------------

    "Tell me about Mars visas."
]

In [85]:
results = []

for question in evaluation_questions:

    print("=" * 80)
    print(question)

    answer = visa_agent(question)

    print(answer)

    results.append({

        "Question": question,

        "Answer": answer

    })

2026-07-25 20:33:09,221 | INFO | Question: What documents are required for a Skilled Worker visa?
INFO:VisaAgent:Question: What documents are required for a Skilled Worker visa?


What documents are required for a Skilled Worker visa?
PLAN
{'steps': ['search_documents', 'answer_directly']}


2026-07-25 20:33:17,748 | INFO | Answer: Based on official GOV.UK guidance, please note that any eligibility check mentioned is **only an initial assessment** and not a final decision. 

To apply for a **Skilled Worker visa**, you will need to provide the following mandatory documents and information:

### Required Documents and Details
* **Valid Passport or Travel Document:** To establish your identity and nationality.
* **Certificate of Sponsorship (CoS) Reference Number:** Provided by your UK employer.
* **Sponsor and Job Details:** 
  * Employer's name and sponsor licence number (found on your CoS)
  * Job title and annual salary
  * Job occupation code
* **Proof of Knowledge of English:** Evidence satisfying the English language requirement (e.g., approved English test result, qualifying degree, or nationality from an exempt country).
* **Payment & Biometrics:** Evidence/confirmation that any required application fee and Immigration Health Charge (IHC) have been paid, and provisio

Response Time: 8.53 seconds
Based on official GOV.UK guidance, please note that any eligibility check mentioned is **only an initial assessment** and not a final decision. 

To apply for a **Skilled Worker visa**, you will need to provide the following mandatory documents and information:

### Required Documents and Details
* **Valid Passport or Travel Document:** To establish your identity and nationality.
* **Certificate of Sponsorship (CoS) Reference Number:** Provided by your UK employer.
* **Sponsor and Job Details:** 
  * Employer's name and sponsor licence number (found on your CoS)
  * Job title and annual salary
  * Job occupation code
* **Proof of Knowledge of English:** Evidence satisfying the English language requirement (e.g., approved English test result, qualifying degree, or nationality from an exempt country).
* **Payment & Biometrics:** Evidence/confirmation that any required application fee and Immigration Health Charge (IHC) have been paid, and provision of biometri

2026-07-25 20:33:31,722 | INFO | Answer: Please note that any eligibility check mentioned is **only an initial assessment**.

Based on official GOV.UK guidance, here are the details regarding bringing your spouse and your study/work permissions:

---

### 1. Can my spouse come with me on a Skilled Worker visa?

**Yes**, you can bring your spouse (partner) as your dependant, provided eligibility requirements are met:

* **Entry Clearance Requirement (SW 28.1):** Your spouse must apply online and obtain entry clearance as a dependent partner before traveling to the UK.
* **Tuberculosis (TB) Screening (SW 28.2):** If Appendix Tuberculosis applies based on their country of residence, they must provide a valid medical certificate confirming they do not have active pulmonary tuberculosis.
* **Financial / Maintenance Evidence:** Your spouse will usually need to show proof of funds to support themselves when applying, **unless**:
  * You have all been living in the UK with a valid visa for at 

...

Response Time: 12.8 seconds
Please note that any eligibility check mentioned is **only an initial assessment**.

Based on official GOV.UK guidance, the documents required for a Visitor visa depend on your circumstances and the specific activities you plan to undertake. The guidance sets out mandatory evidence for certain types of visitors, as well as recommended supporting documents to satisfy the decision-maker that you are a genuine visitor.

### Supporting and Financial Documents You May Need
* **Financial Evidence:** Documents showing you have sufficient funds available, such as:
  * Bank statements detailing the origin of the funds held.
  * Building society books detailing the origin of funds held.
* **Proof of Employment or Earnings:** A letter from your employer confirming your employment details (including your start date, salary, job role, and company contact details).
* **Proof of Self-Employment:** Business registration documents or recent invoices confirming ongoing self-e

2026-07-25 20:34:11,889 | INFO | Answer: I am a UK Visa Guidance Assistant, so I assist with UK visa, immigration, and entry requirement questions based on official GOV.UK guidance. I do not have access to real-time weather forecasts or live data for London. 

For current weather information, please check a reliable weather service or forecasting website. If you have any questions regarding UK visas or immigration requirements, feel free to ask!
INFO:VisaAgent:Answer: I am a UK Visa Guidance Assistant, so I assist with UK visa, immigration, and entry requirement questions based on official GOV.UK guidance. I do not have access to real-time weather forecasts or live data for London. 

For current weather information, please check a reliable weather service or forecasting website. If you have any questions regarding UK visas or immigration requirements, feel free to ask!
2026-07-25 20:34:11,891 | INFO | Latency: 5.72 seconds
INFO:VisaAgent:Latency: 5.72 seconds
2026-07-25 20:34:11,893 | 

Response Time: 5.72 seconds
I am a UK Visa Guidance Assistant, so I assist with UK visa, immigration, and entry requirement questions based on official GOV.UK guidance. I do not have access to real-time weather forecasts or live data for London. 

For current weather information, please check a reliable weather service or forecasting website. If you have any questions regarding UK visas or immigration requirements, feel free to ask!
Tell me about Mars visas.
PLAN
{'steps': ['answer_directly']}


2026-07-25 20:34:19,314 | INFO | Answer: Please note that any eligibility check mentioned is **only an initial assessment**.

Based on official GOV.UK guidance, there is no visa route or guidance available for "Mars visas." Mars is a planet, and the UK Home Office only issues visas and entry clearance for individuals traveling to or staying within the United Kingdom. 

If you have questions about actual UK visa categories (such as Skilled Worker, Student, or Visitor visas) or UK immigration requirements, please feel free to ask!
INFO:VisaAgent:Answer: Please note that any eligibility check mentioned is **only an initial assessment**.

Based on official GOV.UK guidance, there is no visa route or guidance available for "Mars visas." Mars is a planet, and the UK Home Office only issues visas and entry clearance for individuals traveling to or staying within the United Kingdom. 

If you have questions about actual UK visa categories (such as Skilled Worker, Student, or Visitor visas) or UK

Response Time: 7.42 seconds
Please note that any eligibility check mentioned is **only an initial assessment**.

Based on official GOV.UK guidance, there is no visa route or guidance available for "Mars visas." Mars is a planet, and the UK Home Office only issues visas and entry clearance for individuals traveling to or staying within the United Kingdom. 

If you have questions about actual UK visa categories (such as Skilled Worker, Student, or Visitor visas) or UK immigration requirements, please feel free to ask!


In [94]:
# First response
print(visa_agent(
    "What documents are needed for a Student visa?"
))

# User feedback
store_feedback(

    question="What documents are needed for a Student visa?",

    answer="...",

    rating="negative",

    comments="Please answer briefly with bullet points."
)

# Same question again
print(visa_agent(
    "What documents are needed for a Student visa?"
))


2026-07-25 22:14:08,168 | INFO | Question: What documents are needed for a Student visa?
INFO:VisaAgent:Question: What documents are needed for a Student visa?


PLAN
{'steps': ['search_documents', 'answer_directly']}


2026-07-25 22:14:15,868 | INFO | Answer: Based on official GOV.UK guidance, here are the required documents and evidence for a Student visa:

**Required for all applicants:**
* A current valid passport or travel document (used via the 'UK Immigration: ID Check' app or brought to a visa application centre)
* Proof of a course offer from a licensed student sponsor (Confirmation of Acceptance for Studies / CAS)
* Proof of sufficient funds to support yourself and pay for your course
* Proof of English language ability (speaking, reading, writing, and understanding)

**Additional documents (depending on your circumstances):**
* Written consent from your financial sponsor if you received sponsorship for course fees and living costs in the last 12 months

**If you are 16 or 17 years old (under 18):**
* Written consent from both parents or legal guardians (or one parent if sole responsibility) covering:
  * Your visa application
  * Your living and care arrangements in the UK
  * Your travel t

Response Time: 7.7 seconds
Based on official GOV.UK guidance, here are the required documents and evidence for a Student visa:

**Required for all applicants:**
* A current valid passport or travel document (used via the 'UK Immigration: ID Check' app or brought to a visa application centre)
* Proof of a course offer from a licensed student sponsor (Confirmation of Acceptance for Studies / CAS)
* Proof of sufficient funds to support yourself and pay for your course
* Proof of English language ability (speaking, reading, writing, and understanding)

**Additional documents (depending on your circumstances):**
* Written consent from your financial sponsor if you received sponsorship for course fees and living costs in the last 12 months

**If you are 16 or 17 years old (under 18):**
* Written consent from both parents or legal guardians (or one parent if sole responsibility) covering:
  * Your visa application
  * Your living and care arrangements in the UK
  * Your travel to the UK
* A c

2026-07-25 22:14:24,386 | INFO | Answer: Based on official GOV.UK guidance, here are the required documents and evidence for a Student visa:

**Required for all applicants:**
* A current valid passport or travel document (scanned using the 'UK Immigration: ID Check' app or taken to a visa application centre)
* Proof of a course offer from a licensed student sponsor (Confirmation of Acceptance for Studies / CAS)
* Proof of sufficient money to support yourself and pay for your course
* Proof of English language ability (speaking, reading, writing, and understanding)

**Additional documents (depending on your circumstances):**
* Written consent from your financial sponsor if you received sponsorship for course fees and living costs in the last 12 months

**If you are 16 or 17 years old (under 18):**
* Written consent from both parents or legal guardians (or one parent if sole responsibility) covering:
  * Your visa application
  * Your living and care arrangements in the UK
  * Your trave

Response Time: 8.52 seconds
Based on official GOV.UK guidance, here are the required documents and evidence for a Student visa:

**Required for all applicants:**
* A current valid passport or travel document (scanned using the 'UK Immigration: ID Check' app or taken to a visa application centre)
* Proof of a course offer from a licensed student sponsor (Confirmation of Acceptance for Studies / CAS)
* Proof of sufficient money to support yourself and pay for your course
* Proof of English language ability (speaking, reading, writing, and understanding)

**Additional documents (depending on your circumstances):**
* Written consent from your financial sponsor if you received sponsorship for course fees and living costs in the last 12 months

**If you are 16 or 17 years old (under 18):**
* Written consent from both parents or legal guardians (or one parent if sole responsibility) covering:
  * Your visa application
  * Your living and care arrangements in the UK
  * Your travel to the UK
*

In [96]:
print(visa_agent("What is the weather in London today?"))

2026-07-25 22:31:52,230 | INFO | Question: What is the weather in London today?
INFO:VisaAgent:Question: What is the weather in London today?


PLAN
{'steps': ['answer_directly']}


2026-07-25 22:31:58,203 | INFO | Answer: No relevant GOV.UK guidance found for London weather.

* The UK Visa and Immigration guidance covers entry requirements, visas, and citizenship rules, not current weather or climate forecasts.
* For official UK weather updates, please consult the UK Met Office.

***

**Sources:**
* GOV.UK: Visas and immigration (`/browse/visas-immigration`)
INFO:VisaAgent:Answer: No relevant GOV.UK guidance found for London weather.

* The UK Visa and Immigration guidance covers entry requirements, visas, and citizenship rules, not current weather or climate forecasts.
* For official UK weather updates, please consult the UK Met Office.

***

**Sources:**
* GOV.UK: Visas and immigration (`/browse/visas-immigration`)
2026-07-25 22:31:58,205 | INFO | Latency: 5.97 seconds
INFO:VisaAgent:Latency: 5.97 seconds


Response Time: 5.97 seconds
No relevant GOV.UK guidance found for London weather.

* The UK Visa and Immigration guidance covers entry requirements, visas, and citizenship rules, not current weather or climate forecasts.
* For official UK weather updates, please consult the UK Met Office.

***

**Sources:**
* GOV.UK: Visas and immigration (`/browse/visas-immigration`)


# Summary
This notebook demonstrates an AI agent that combines retrieval-augmented generation, tool use, planning, short-term memory and adaptive behaviour to answer UK visa queries using official GOV.UK guidance.